In [1]:
%load_ext autoreload
%autoreload 2
from mom6_bathy import mapping, plotting
import xarray as xr
import numpy as np

In [32]:
mapping.generate_ESMF_map_via_esmpy(
    src_mesh_path='/glade/p/cesmdata/cseg/inputdata/share/meshes/rx1_nomask_181022_ESMFmesh.nc',
    dst_mesh_path='/glade/p/cesmdata/cseg/inputdata/share/meshes/tx2_3v2_230415_ESMFmesh.nc',
    mapping_file='/glade/derecho/scratch/manishrv/croc_map_rx1_to_t232_v4.nc',
    method='nearest_s2d',
    area_normalization=True,
)

In [ ]:

import os
import xesmf as xe




In [31]:
mapping_file='/glade/derecho/scratch/manishrv/xesmf_map.nc'
src_mesh_path='/glade/p/cesmdata/cseg/inputdata/share/meshes/rx1_nomask_181022_ESMFmesh.nc'
dst_mesh_path='/glade/p/cesmdata/cseg/inputdata/share/meshes/tx2_3v2_230415_ESMFmesh.nc'
src_mesh_ds = xr.open_dataset(src_mesh_path)
dst_mesh_ds = xr.open_dataset(dst_mesh_path)
area_normalization = True
method = "nearest_s2d"
# Run the regridder
if os.path.exists(mapping_file):
    os.remove(mapping_file)

# Create regridder and save the weights to the mapping file temporarily
# This file will later be extended to include the full ESMF map fields
src_mesh_ds = xr.open_dataset(src_mesh_path)
dst_mesh_ds = xr.open_dataset(dst_mesh_path)
src_grid = mapping.grid_from_esmf_mesh(src_mesh_ds) # Mask is in here
dst_grid = mapping.grid_from_esmf_mesh(dst_mesh_ds) # Mask is in here

xe.Regridder(
    src_grid,
    dst_grid,
    method=method,
    filename=mapping_file,
   
)

# Now, read in the weights from the mapping file
weights_ds = xr.open_dataset(mapping_file)

mapping.generate_ESMF_map(
    src_mesh=src_mesh_path,
    dst_mesh=dst_mesh_path,
    filename=mapping_file,
    weights=weights_ds,
    weights_esmpy = None,
    area_normalization=area_normalization
)

/glade/work/manishrv/conda-envs/CrocoDash/lib/python3.12/site-packages/xesmf/backend.py:41: UserWarning: Input array is not F_CONTIGUOUS. Will affect performance.
  warnings.warn('Input array is not F_CONTIGUOUS. ' 'Will affect performance.')


In [33]:
# Read Mapping File
xesmf_map = xr.open_dataset("/glade/derecho/scratch/manishrv/xesmf_map.nc")
esmpy_map = xr.open_dataset("/glade/derecho/scratch/manishrv/croc_map_rx1_to_t232_v4.nc")

In [34]:
esmpy_map

<xarray.Dataset>
Dimensions:        (n_a: 64800, nv_a: 4, src_grid_rank: 2, nj_a: 180,
                    ni_a: 360, n_b: 259200, nv_b: 4, dst_grid_rank: 2,
                    nj_b: 480, ni_b: 540, n_s: 152349)
Coordinates:
  * nj_a           (nj_a) int32 1 2 3 4 5 6 7 8 ... 174 175 176 177 178 179 180
  * ni_a           (ni_a) int32 1 2 3 4 5 6 7 8 ... 354 355 356 357 358 359 360
  * nj_b           (nj_b) int32 1 2 3 4 5 6 7 8 ... 474 475 476 477 478 479 480
  * ni_b           (ni_b) int32 1 2 3 4 5 6 7 8 ... 534 535 536 537 538 539 540
Dimensions without coordinates: n_a, nv_a, src_grid_rank, n_b, nv_b,
                                dst_grid_rank, n_s
Data variables: (12/19)
    xc_a           (n_a) float64 ...
    yc_a           (n_a) float64 ...
    xv_a           (n_a, nv_a) float64 ...
    yv_a           (n_a, nv_a) float64 ...
    mask_a         (n_a) int32 ...
    area_a         (n_a) float64 ...
    ...             ...
    area_b         (n_b) float64 ...
    frac_b         (n_b) float64 ...
    dst_grid_dims  (dst_grid_rank) int32 ...
    S              (n_s) float64 ...
    col            (n_s) int32 ...
    row            (n_s) int32 ...
Attributes:
    title:        mom6_bathy mapping generation
    history:      generated by mom6_bathy
    conventions:  NCAR-CCSM

In [ ]:
for i in range(len(esmpy_map.n_s)):
    esmpy_row_num = esmpy_map.row[i].item()
    search = np.where(xesmf_map.row == esmpy_row_num)[0]
    if len(search)>1:
        raise ValueError()
    xesmf_row = xesmf_map.row[search[0]].item()
    esmpy_row = esmpy_map.row[i].item()
    assert xesmf_row == esmpy_row


ValueError: 

In [11]:
coastline_mask = mapping.extract_coastline_mask(grid)
np.where(coastline_mask!=0)

(array([ 26,  26,  26, ..., 479, 479, 479]),
 array([182, 184, 185, ..., 473, 474, 475]))

In [12]:
src_mesh_ds

<xarray.Dataset>
Dimensions:         (nodeCount: 65160, coordDim: 2, elementCount: 64800,
                     maxNodePElement: 4)
Dimensions without coordinates: nodeCount, coordDim, elementCount,
                                maxNodePElement
Data variables:
    nodeCoords      (nodeCount, coordDim) float64 ...
    elementConn     (elementCount, maxNodePElement) float64 ...
    numElementConn  (elementCount) int8 ...
    centerCoords    (elementCount, coordDim) float64 -179.5 -89.5 ... 179.5 89.5
    elementArea     (elementCount) float64 ...
    elementMask     (elementCount) int32 1 1 1 1 1 1 1 1 1 ... 1 1 1 1 1 1 1 1 1
Attributes:
    gridType:       unstructured mesh
    version:        0.9
    inputFile:      1degcore_130703.nc
    timeGenerated:  Mo